# Agent architecture diagrams

Draws the two LangGraph pipelines from `sql_agent_pipeline.py` — **agent_1** (simple) and **agent_2**
(relevance check + validation + retry) — and saves each as PNG (300 dpi), SVG and PDF in `docs/figures/` (the repository keeps only the two LangGraph PNGs; run the notebook to regenerate the rest).

The graphs are read from the real compiled agents (`app.get_graph()`), so the nodes and edges shown are the
ones the benchmark runs. No database or API key is needed: the LLM and database are never called when the
graph is only drawn.

1. Build both agents.
2. Optional: LangGraph's own Mermaid rendering (needs internet).
3. Publication figure drawn with matplotlib (works offline).

In [ ]:
import sys
from pathlib import Path

# This notebook lives in docs/; the agent code is in the repository root.
REPO_ROOT = Path.cwd().parent if (Path.cwd().parent / "sql_agent_pipeline.py").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from prompt_stages import STAGE_PROMPTS
from sql_agent_pipeline import AGENTS

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# llm=None, db=None is fine: the node functions are never executed, only the graph structure is read.
apps = {name: build(STAGE_PROMPTS["stage1_simple"], None, None) for name, build in AGENTS.items()}
graphs = {name: app.get_graph() for name, app in apps.items()}

for name, g in graphs.items():
    print(name, "nodes:", [n for n in g.nodes])
    for e in g.edges:
        print("   ", e.source, "->", e.target, "(conditional)" if e.conditional else "")

## LangGraph's built-in rendering (optional)

Uses the mermaid.ink web service, so it needs an internet connection. If it fails, skip to the next section.

In [ ]:
from IPython.display import Image, display

for name, g in graphs.items():
    try:
        png = g.draw_mermaid_png()
    except Exception as exc:                      # no internet / service unavailable
        print(f"{name}: Mermaid rendering unavailable ({exc})")
        continue
    (FIG_DIR / f"{name}_langgraph.png").write_bytes(png)
    print(name)
    display(Image(png))

## Publication figure (matplotlib)

Blue boxes are steps that call the LLM; green boxes are database or rule-based steps (the relevance checker is a keyword filter that only calls the LLM to word a refusal). Solid arrows are fixed
edges, dashed arrows are conditional ones (labelled with the condition). Node order and labels are set in
`ORDER` and `LABELS`; edges come from the compiled graph.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle

LLM_NODES = {"query_generator", "answer_synthesizer"}   # the relevance checker is a keyword filter
LABELS = {
    "__start__": "START",
    "__end__": "END",
    "relevance_checker": "Relevance\nchecker",
    "schema_fetcher": "Schema\nfetcher",
    "query_generator": "Query\ngenerator",
    "query_validator": "Query\nvalidator\n(EXPLAIN)",
    "query_runner": "Query\nrunner",
    "answer_synthesizer": "Answer\nsynthesizer",
}
ORDER = {   # left-to-right position of each node
    "agent_1": ["__start__", "schema_fetcher", "query_generator", "query_runner", "answer_synthesizer", "__end__"],
    "agent_2": ["__start__", "relevance_checker", "schema_fetcher", "query_generator", "query_validator",
                "query_runner", "answer_synthesizer", "__end__"],
}
# Conditional edges that skip over or go back past a neighbour are drawn as arcs: (source, target) -> (label, arc height)
ARCS = {
    ("relevance_checker", "__end__"): ("not relevant: stop", 1.7),
    ("query_validator", "answer_synthesizer"): ("3 failed attempts", 0.95),
    ("query_validator", "query_generator"): ("invalid SQL: retry (max 3)", 0.95),
}
TITLES = {
    "agent_1": "agent_1: simple pipeline (no relevance check, validation or retry)",
    "agent_2": "agent_2: relevance check, EXPLAIN validation and up to 3 retries",
}
COLORS = {"llm": ("#dbeafe", "#1d4ed8"), "tool": ("#dcfce7", "#15803d"), "end": ("#f3f4f6", "#4b5563")}
STEP, BOX_W, BOX_H = 2.3, 1.55, 1.1
EDGE_COLOR = "#374151"


def kind(node):
    return "llm" if node in LLM_NODES else "end" if node.startswith("__") else "tool"


def draw_agent(name, graph, save=True):
    order = ORDER[name]
    assert set(order) == set(graph.nodes), (set(order) ^ set(graph.nodes))
    xs = {n: i * STEP for i, n in enumerate(order)}
    arcs = {k: v for k, v in ARCS.items() if k in {(e.source, e.target) for e in graph.edges}}
    top = max([h for k, (_, h) in arcs.items() if order.index(k[1]) > order.index(k[0])], default=0)
    bottom = max([h for k, (_, h) in arcs.items() if order.index(k[1]) < order.index(k[0])], default=0)
    fig, ax = plt.subplots(figsize=(xs[order[-1]] + 2.4, 1.9 + top + bottom))

    boxes = {}
    for n in order:
        face, edge = COLORS[kind(n)]
        w = 1.0 if kind(n) == "end" else BOX_W
        box = FancyBboxPatch((xs[n] - w / 2, -BOX_H / 2), w, BOX_H, boxstyle="round,pad=0.02,rounding_size=0.18",
                             fc=face, ec=edge, lw=1.6)
        ax.add_patch(box)
        ax.text(xs[n], 0, LABELS[n], ha="center", va="center", fontsize=10, color="#111827", linespacing=1.25)
        boxes[n] = box

    for e in graph.edges:
        i, j = order.index(e.source), order.index(e.target)
        arc = ARCS.get((e.source, e.target))
        if arc:
            label, height = arc
            d = abs(xs[e.target] - xs[e.source])
            rad = -2 * height / d                       # arc apex sits `height` above (forward) or below (back)
            style = f"arc3,rad={rad}"
        else:
            assert j == i + 1, f"edge {e.source}->{e.target} needs an ARCS entry"
            style = "arc3,rad=0"
        ax.add_patch(FancyArrowPatch(
            (xs[e.source], 0), (xs[e.target], 0), patchA=boxes[e.source], patchB=boxes[e.target],
            arrowstyle="-|>", mutation_scale=15, lw=1.6, color=EDGE_COLOR,
            linestyle="--" if e.conditional else "-", connectionstyle=style, shrinkA=0, shrinkB=0))
        if arc:
            sign = 1 if j > i else -1
            ax.text((xs[e.source] + xs[e.target]) / 2, sign * (height + 0.1), label, ha="center",
                    va="bottom" if sign > 0 else "top", fontsize=9, style="italic", color=EDGE_COLOR)

    ax.set_xlim(-0.8, xs[order[-1]] + 0.8)
    ax.set_ylim(-max(bottom + 0.55, BOX_H / 2 + 0.15), max(top + 0.55, BOX_H / 2 + 0.15))
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(TITLES[name], fontsize=12, pad=8)
    ax.legend(handles=[Rectangle((0, 0), 1, 1, fc=COLORS["llm"][0], ec=COLORS["llm"][1], label="LLM call"),
                       Rectangle((0, 0), 1, 1, fc=COLORS["tool"][0], ec=COLORS["tool"][1],
                                 label="database / rule-based step"),
                       Line2D([0], [0], color=EDGE_COLOR, lw=1.6, label="fixed edge"),
                       Line2D([0], [0], color=EDGE_COLOR, lw=1.6, ls="--", label="conditional edge")],
              loc="upper center", bbox_to_anchor=(0.5, 0.0), frameon=False, fontsize=9, ncol=4)
    fig.tight_layout()
    if save:
        for ext in ("png", "svg", "pdf"):
            fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
    return fig


fig1 = draw_agent("agent_1", graphs["agent_1"])
plt.show()

In [ ]:
fig2 = draw_agent("agent_2", graphs["agent_2"])
plt.show()

## Download

The files are written to the `figures/` folder next to this notebook (`docs/figures/`): `agent_1.png`, `agent_2.png` (300 dpi),
plus `.svg` / `.pdf` versions that stay sharp at any size (best for LaTeX or Word). The links below open them.
In Google Colab, run the last cell to download.

In [ ]:
from IPython.display import FileLink, display

for f in sorted(FIG_DIR.glob("agent_*.*")):
    display(FileLink(str(f)))

# Google Colab only:
# from google.colab import files
# for f in FIG_DIR.glob("agent_?.png"):
#     files.download(str(f))